#### Modern RNN Units:
- LSTM — Long Short‑Term Memory
- GRU — Gated Recurrent Unit
- GRU is like a simplified version of the LSTM (fewer parameters → more efficient)

#### Why do we need these at all? hy do we need fancy i.e Mordern RNNs?

#### 1. The Nesting Problem
A Simple RNN is essentially a massive nested composite function. As the sequence length $T$ increases, the early inputs ($x_1$) become buried deep inside the equation.

**The "Deeply Nested" Equation:**
$$\hat{y}(T) = \sigma(W_o^T \sigma(W_{xh}^T x_T + W_{hh}^T \sigma(W_{xh}^T x_{T-1} + W_{hh}^T \sigma(...(W_{xh}^T x_1 + W_{hh}^T h_0...)))))$$

*   $x_1$ is the most deeply nested term.
*   $x_T$ is the least nested.

**The Nesting Equation:**

This equation visualizes the unrolled version of the RNN. Every time you see a $\sigma(\dots)$, that represents one "tick" of the hidden state calculation. Notice how $x_{1}$ is inside the innermost parentheses. To reach $x_{1}$, the signal has to travel through every single weight matrix $W_{hh}$ and activation function $\sigma $ in the entire sequence.

#### 2. The Vanishing Gradient Problem
When training via Backpropagation Through Time (BPTT), we need the gradient of the loss $J$ with respect to the weights $W_{xh}$. This gradient is a function of the individual gradients from every time step:

$$\frac{\partial J}{\partial W_{xh}} = f\left(\frac{\partial(W_{xh}^T x_T)}{\partial W_{xh}}, \frac{\partial(W_{xh}^T x_{T-1})}{\partial W_{xh}}, \dots, \frac{\partial(W_{xh}^T x_1)}{\partial W_{xh}}\right)$$

*   **The Chain Rule:** Because the function is so nested, calculating the gradient for $x_1$ requires multiplying many derivatives together.
*   **The Result:** If these derivatives are small (which they often are with activation functions like sigmoid or tanh), the gradient "vanishes" (becomes near zero) by the time it reaches $x_1$.
*   **Consequence:** Simple RNNs "forget" long-term dependencies because they can't effectively update weights based on inputs from far back in the sequence.

**The Gradient Equation:**

This equation simplifies the goal of training. To update our weights ($W_{xh}$), we look at how those weights affected the output at every single time step.The gradient for $x_{T}$ (the most recent input) is easy to calculate because it's "on the surface."The gradient for $x_{1}$ is extremely difficult to calculate because it is at the bottom of the "nest."

#### Additinals understanding about Vanishing Gradient Problem:

Take this example:

![Albert Einstein](image-20.png)

Ask your simple RNN: “Suppose our AI is reading the Wikipedia page about Albert Einstein On what day was Albert Einstein born?”

By the time the RNN has “read” the whole article, it's read the entire page. Now, you ask your AI on what day was Albert Einstein born? Unfortunately, a simple RNN would not be able to answer this
even though this information appears in the article. The problem is it appeared at the beginning of the article, so by the end of the article, the simple RNN has simply forgotten **what it had read earlier**.”

#### In ANN architecture I know relu activations functions. The derivative of RELU are the = 1. As in back propagations derivative are multiplied. Multiplications with const is always const.How it help to avoided gradient vanishing and exploding problem?

- **The Gradient Problem Recap**

* **Vanishing gradients**: In sigmoid/tanh, derivatives are bounded $0 < \sigma'(x) < 1$. Multiplying many small numbers across layers shrinks gradients $\rightarrow$ weights stop updating.
* **Exploding gradients**: In deep nets, multiplying large derivatives can blow up gradients $\rightarrow$ unstable updates.

- **Why ReLU Helps?**

1. **Derivative is 1 (not $< 1$)**
   * For $x > 0$,
   
     $$\frac{d}{dx}\text{ReLU}(x) = 1$$
     
   * This means gradients **don't shrink** as they propagate through layers. Unlike sigmoid/tanh, you're not multiplying by fractions repeatedly.

2. **Piecewise linearity**
   * ReLU is linear for positive inputs, so backpropagation behaves like passing gradients through a linear chain.
   * This avoids exponential decay of gradients.

3. **Sparse activation (0 for negatives)**
   * For $x < 0$, $\text{derivative} = 0 \rightarrow$ neuron doesn't update.
   * This sparsity reduces the risk of exploding gradients because not all neurons fire simultaneously.

4. **Scaling stability**
   * With proper initialization (He initialization), the variance of activations stays stable across layers.
   * Combined with $\text{derivative} = 1$, this keeps gradients well-scaled.

Think of gradient flow as water through pipes:

* **Sigmoid/tanh**: Narrow pipes $\rightarrow$ flow shrinks at each layer (vanishing).

![Sigmoid Tanh](image-24.png)

- Each derivative is a fraction (<1).
- Multiplying many fractions → gradient decays exponentially.
- Deep networks stop learning (vanishing gradient problem).

* **ReLU**: Straight pipes with constant width $\rightarrow$ flow passes through without shrinking.

![ReLU](image-25.png)

- For positive inputs, derivative = 1 → gradient flows without shrinking.
- For negative inputs, derivative = 0 → neuron shuts off (sparse activation).
- This avoids vanishing and reduces exploding risk.

* **Occasional "blocked pipes" (zeros)** prevent overflow (exploding).
That’s why ReLU became the default activation in deep learning: it keeps gradients stable across many layers, unlike sigmoid/tanh which cause vanishing.

#### Does ReLU solve the Vanishing Gradient Problem in RNNs? Just use ReLU?

![RELU](image-21.png)

*   **Unfortunately not so simple.**
*   But you can always try and experiment!
*   In Deep Learning, researchers have discovered that **GRUs** and **LSTMs** are more effective.

The short answer is **no**, not entirely. While ReLU helps a lot in Feedforward networks (like CNNs), it introduces a new set of problems in RNNs:

##### 1. Activation vs. Weights
The Vanishing Gradient Problem is caused by two things being multiplied repeatedly: 
1.  The derivative of the **activation function** ($\sigma'$).
2.  The **Recurrent Weight Matrix** ($W_{hh}$).

ReLU solves the first part because its derivative is **1** (for positive values), so it doesn't "shrink" the signal like Sigmoid or Tanh does. However, the $W_{hh}$ matrix is still multiplied over and over at every time step.

##### 2. The "Exploding" Risk
Because the derivative of ReLU is 1, if your weights ($W_{hh}$) are even slightly greater than 1, the gradient will **explode** (grow to infinity) very quickly as you move back through time. 
*   **Sigmoid/Tanh:** Usually vanish (gradient $\rightarrow$ 0).
*   **ReLU:** Tends to explode (gradient $\rightarrow$ $\infty$) or stay at 0 (if inputs are negative).

#### Why mordern RNN i.e. LSTMs/GRUs are better

"Fancy RNNs" (like LSTMs or GRUs) were invented specifically to solve this. Instead of just changing the activation function, **LSTMs** and **GRUs** change the **architecture**. They use "gates" to create a linear "highway" for the gradient to flow through without being forced through a repetitive multiplication of $W_{hh}$ at every step.

# GRUs (Gated Recurrent Unit)
## [Go for “interesting”, not for “popular”]

Created in 1997 (even before the field was called “deep learning”)

![GRU](image-22.png)


Diagram 1 (SimpleRNN):

*   **Inputs:** $x(t), h(t-1)$
*   **Output:** $h(t)$

Diagram 2 (GRU):

*   **Inputs:** $x(t), h(t-1)$
*   **Output:** $h(t)$

#### Digram and equations:

![Gated Recurrent Unit (GRU)](image-23.png)

#### Gated Recurrent Unit (GRU)

The GRU is an improvement over the Simple RNN designed to solve the **vanishing gradient problem** by using "gates" to control the flow of information.

##### 1. Simple RNN Unit (For Comparison)
In a standard RNN, the hidden state is completely overwritten at every step:

$$h_t = \tanh(W_{xh}^T x_t + W_{hh}^T h_{t-1} + b_h)$$

##### 2. The GRU Architecture
The GRU introduces two vectors that act as "filters" or "switches":
*   **$z_t$ (Update Gate):** Determines how much of the *past* state to keep vs. how much *new* information to add.
    - The "Gating" Mechanism: The use of the Sigmoid ($\sigma $) function for $z_{t}$ and $r_{t}$ is critical. Since Sigmoid outputs values between 0 and 1, it acts like a valve. A "0" closes the valve (stops information), and a "1" opens it.
    - Solving the Vanishing Gradient: In a Simple RNN, gradients must pass through a $\tanh $ and a weight matrix multiplication at every step, causing them to shrink. In a GRU, if $z_{t}$ is near 0, the gradient can slide through the $(1-z_t)h_{t-1}$ part of the equation almost unchanged, allowing the "memory" to stay alive for hundreds of steps.
*   **$r_t$ (Reset Gate):** Determines how much of the *past* state to forget before calculating the new candidate state.
    - The Reset Gate ($r_{t}$): This gate is inside the $\tanh $. It allows the model to decide if the previous hidden state is actually relevant for the current calculation. If $r_{t}$ is 0, it wipes the slate clean for that specific time step's update.
    

##### Core Equations:
1. **Update Gate:** $z_t = \sigma(W_{xz}^T x_t + W_{hz}^T h_{t-1} + b_z)$
2. **Reset Gate:** $r_t = \sigma(W_{xr}^T x_t + W_{hr}^T h_{t-1} + b_r)$
3. **Hidden State:** $h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tanh(W_{xh}^T x_t + W_{hh}^T (r_t \odot h_{t-1}) + b_h)$

> **Note:** $\odot$ denotes element-wise multiplicatio

##### 3. Interpreting the Logic
Think of the final $h_t$ equation as a **weighted average** between the old state and a new "Simple RNN" style update:

$$h_t \approx p(\text{keep } h_{t-1})h_{t-1} + p(\text{discard } h_{t-1})\text{SimpleRNN}(x_t, h_{t-1})$$

*   **If $z_t \approx 0$:** The model "ignores" the new input and holds onto the old memory ($h_{t-1}$). This is how it remembers things from far in the past!
*   **If $z_t \approx 1$:** The model "forgets" the old state and replaces it with the new calculation.

##### 4. Implementation Shortcut: "Return of the Neuron"
Calculating $z_t$ is essentially just **Logistic Regression** (a single neuron with a sigmoid activation). To make the code faster, we often "stack" the inputs and weights:

$$v = [x_t; h_{t-1}], \quad W_{vz} = [W_{xz}; W_{hz}]$$
$$z_t = \sigma(W_{vz}^T v + b_z)$$

#### GRU: The Reset Gate and Sliding Door Analogy

##### 1. The "Sliding Door" Analogy
This analogy explains the relationship between the **Update Gate ($z_t$)** and the hidden states.

*   **The Rule:** The gate is the size of exactly one opening. 
*   **The Trade-off:** If you open the door to let in more "new information" (the candidate state), you must proportionally close the door on the "old information" ($h_{t-1}$).
*   **The Math:** This is why we use $(1 - z_t)$ for the old state and $z_t$ for the new state. Their sum is always **100%**.
    *   *Example:* If $z_t = 0.75$, you are keeping 25% of the past and mixing it with 75% of the new update.

##### 2. The Reset Gate ($r_t$)
The reset gate is just another logistic regression neuron ($\sigma$). Its job is to decide how much of the past hidden state is actually useful for the *current* calculation.

**The Equation:**
$$r_t = \sigma(W_{xr}^T x_t + W_{hr}^T h_{t-1} + b_r)$$

**How it fits into the Hidden State ($h_t$):**
$$h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tanh(W_{xh}^T x_t + W_{hh}^T (\mathbf{r_t \odot h_{t-1}}) + b_h)$$

*   **The "Switch":** If $r_t$ is near 0, the model "forgets" the past $h_{t-1}$ specifically for this current $\tanh$ calculation, effectively resetting the unit.

The **Sliding Door** Analogy Think of the hidden state as a room with a limited capacity. You only have enough "space" for one full state. If the model decides that the current input $x_{t}$ is very important (high $z_{t}$), it has to "slide the door" to let that new info in, which naturally pushes out an equivalent amount of the old memory. This ensures the signal doesn't explode or grow out of control. The **Reset Gate** While the Update Gate ($z_{t}$) decides what stays in the long-term memory, the Reset Gate ($r_{t}$) decides if the past is relevant to the immediate next step.Analogy: If you are reading a book and start a brand-new chapter, your "Reset Gate" might trigger to clear out the specific details of the last scene so you can focus entirely on the new setting, while your "Update Gate" keeps the overall plot (the long-term state) alive.

##### 3. GRU Summary
*   **Same API:** It uses the same inputs ($x_t, h_{t-1}$) and produces the same output ($h_t$) as a Simple RNN.
*   **Better Memory:** Unlike Simple RNNs, which suffer from **vanishing gradients** and forget things quickly, GRUs use these gates to explicitly remember or forget specific components.

# LSTM Cell:(Long Short-Term Memory (LSTM))

Follow thisd LINK -> [LSTM](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)

![LSTM](image-26.png)

“Understanding LSTMs” explains why Long Short‑Term Memory networks solve the vanishing gradient problem in RNNs, and it walks through the internal gates (forget, input, output) and the cell state. The key idea is that LSTMs maintain a “conveyor belt” of memory, selectively updating or forgetting information, which allows them to capture long‑term dependencies in sequences.

Standard recurrent networks pass hidden states forward, but struggle with long‑term dependencies because gradients vanish/explode. Introduced by Hochreiter & Schmidhuber (1997), LSTMs add a cell state and gates to regulate information flow.


The LSTM (Long Short-Term Memory) is a more sophisticated version of the GRU. While the GRU combines its memories into one hidden state, the LSTM splits them into two distinct "tracks" to better handle long-term patterns.

##### LSTM Computations

The following equations represent the internal operations of an LSTM cell at a single time step $t$:

##### Gate Controllers & Candidate State
Each of these layers uses the current input $\mathbf{x}_{(t)}$ and the previous short-term state $\mathbf{h}_{(t-1)}$.

*   **Input gate:** $\mathbf{i}_{(t)} = \sigma(\mathbf{W}_{xi}^T \cdot \mathbf{x}_{(t)} + \mathbf{W}_{hi}^T \cdot \mathbf{h}_{(t-1)} + \mathbf{b}_i)$
*   **Forget gate:** $\mathbf{f}_{(t)} = \sigma(\mathbf{W}_{xf}^T \cdot \mathbf{x}_{(t)} + \mathbf{W}_{hf}^T \cdot \mathbf{h}_{(t-1)} + \mathbf{b}_f)$
*   **Output gate:** $\mathbf{o}_{(t)} = \sigma(\mathbf{W}_{xo}^T \cdot \mathbf{x}_{(t)} + \mathbf{W}_{ho}^T \cdot \mathbf{h}_{(t-1)} + \mathbf{b}_o)$
*   **Candidate state (Main layer):** $\mathbf{g}_{(t)} = \tanh(\mathbf{W}_{xg}^T \cdot \mathbf{x}_{(t)} + \mathbf{W}_{hg}^T \cdot \mathbf{h}_{(t-1)} + \mathbf{b}_g)$

##### State Updates
*   **Long-term state:** $\mathbf{c}_{(t)} = \mathbf{f}_{(t)} \otimes \mathbf{c}_{(t-1)} + \mathbf{i}_{(t)} \otimes \mathbf{g}_{(t)}$
*   **Short-term state (Output):** $\mathbf{y}_{(t)} = \mathbf{h}_{(t)} = \mathbf{o}_{(t)} \otimes \tanh(\mathbf{c}_{(t)})$

##### Key Terms
*   $\mathbf{W}_{xi}, \mathbf{W}_{xf}, \mathbf{W}_{xo}, \mathbf{W}_{xg}$: Weight matrices for the connection to the **input vector** $\mathbf{x}_{(t)}$.
*   $\mathbf{W}_{hi}, \mathbf{W}_{hf}, \mathbf{W}_{ho}, \mathbf{W}_{hg}$: Weight matrices for the connection to the **previous short-term state** $\mathbf{h}_{(t-1)}$.
*   $\mathbf{b}_i, \mathbf{b}_f, \mathbf{b}_o, \mathbf{b}_g$: Bias terms. 
    *   *Note:* TensorFlow initializes $\mathbf{b}_f$ to **1s** instead of 0s to prevent the model from forgetting everything at the start of training.
*   $\otimes$: Element-wise multiplication (Hadamard product).


1. The Two-State System:
- Unlike a Simple RNN, the LSTM manages two separate vectors:
    - $(c_{(t)}$ (Cell State): The "Long-Term Memory." It acts like a conveyor belt that runs straight through the entire sequence with only minor linear interactions. This is the key to solving the vanishing gradient problem.
    - $(h_{(t)}$ (Hidden State): The "Short-Term Memory." This is the version of the memory used for the immediate output and the next step's calculation.

2. Opening the "Box": The Four Layers:
- Inside the LSTM cell, the current input $x_{(t)}$ and previous hidden state $h_{(t-1)}$ are fed into four different fully connected (FC) layers:
    -   **The Main Layer ($g_{(t)}$)**: This uses a tanh activation. It analyzes the current input and previous state to suggest "new memories" to be added.
    - **The Forget Gate ($f_{(t)}$)**: A logistic (sigmoid) layer. It decides which parts of the long-term cell state $c_{(t-1)}$ are no longer useful and should be erased (0 = erase, 1 = keep).
    - **The Input Gate ($i_{(t)}$)**: A logistic layer. It decides which parts of the "new memories" from $g_{(t)}$ are actually worth storing in the long-term state.
    - **The Output Gate ($o_{(t)}$)**: A logistic layer. It decides which parts of the updated long-term memory should be read out to produce the hidden state $h_{(t)}$ for this time step.

3. The Lifecycle of a Memory
    - Step 1 (Forget): The long-term state $c_{(t-1)}$ enters and is filtered by the Forget Gate.
    - Step 2 (Update): New info from the Main Layer is filtered by the Input Gate and then added (using a $+$ sign) to the long-term state. This addition operation is crucial because it allows the gradient to flow through backpropagation without being diminished by multiplication.
    - Step 3 (Output): The updated long-term state $c_{(t)}$ is passed through a tanh function and then filtered by the Output Gate to create the final hidden state $h_{(t)}$.